## Chatbot Evaluation - Using Langsmith

- Gathering Data Points (Model Output vs True Output)
- LLM as a Judge
- Multiple Metric Evaluation (correctness & concision)
- Trying on Multiple Model to select Best Model

In [ ]:
import os
from dotenv import load_dotenv
from langsmith import Client
from langchain_groq import ChatGroq
from langsmith import traceable  # For making a fn / node traceable in Langsmith
from langchain_core.messages import SystemMessage, HumanMessage


load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

In [3]:
client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['7717313f-fc48-4c60-915e-33851d8959bf',
  '68d41171-0d82-4a06-8521-98c19679fbcf',
  '31757db7-a0f6-475d-9fcf-f7f83e027e52',
  'f151880e-7a7b-4b11-8170-30f7b3e4dcfc',
  '0ef059ef-ad48-4ba1-a59d-d2277aab9f12'],
 'count': 5,
 'as_of': '2026-07-27T05:51:49.87541494Z'}

In [ ]:
# Define a Metric - (LLM AS a Judge)

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

# Optional: Add @traceable to make the function itself appear as a trace node
@traceable(name="correctness_evaluator")
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    
    # LangChain passes messages as structured objects
    messages = [
        SystemMessage(content=eval_instructions),
        HumanMessage(content=user_content)
    ]
    
    # Call the LLM using .invoke()
    response = llm.invoke(messages)
    
    # Extract response content and clean whitespace/case matching
    return response.content.strip() == "CORRECT"

In [7]:
# Metric - Concisions (checks whether the actual output is less than 2x the length of the expected result.)

# Adding @traceable lets LangSmith track it as an evaluation metric run.
@traceable(name="concision_metric")
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run Evaluations

In [ ]:
default_instructions = "Respond to the user's question in a short, concise manner (one short sentence)."

@traceable(name="my_app")
def my_app(question: str, model: str = "openai/gpt-oss-120b", instructions: str = default_instructions) -> str:
    
    llm = ChatGroq(
        model=model,
        temperature=0
    )
    
    messages = [
        SystemMessage(content=instructions),
        HumanMessage(content=question),
    ]
    
    response = llm.invoke(messages)
    
    return response.content

def ls_target_fast(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"], model="openai/gpt-oss-120b")}

def ls_target_large(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"], model="openai/gpt-oss-120b")}

In [ ]:
# Evaluation 1
experiment_results_fast = client.evaluate(
    ls_target_fast,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="openai/gpt-oss-120b"  # experiment name
)

View the evaluation results for experiment: 'openai/gpt-oss-120b-43429898' at:
https://apac.smith.langchain.com/o/02fc80ef-23a9-4caa-bf71-3f921d813b58/datasets/9976a9d1-ead8-4af0-b6c8-86de340df27e/compare?selectedSessions=92d7c42e-4e7b-4020-8835-d140df4050a4




5it [00:07,  1.45s/it]


In [ ]:
# Evaluation 2
experiment_results_large = client.evaluate(
    ls_target_large,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="openai/gpt-oss-120b"  # experiment name
)

View the evaluation results for experiment: 'openai/gpt-oss-120b-b560f076' at:
https://apac.smith.langchain.com/o/02fc80ef-23a9-4caa-bf71-3f921d813b58/datasets/9976a9d1-ead8-4af0-b6c8-86de340df27e/compare?selectedSessions=67d2dc0b-3cc9-4861-86b9-0fabd9a4c741




5it [00:07,  1.44s/it]
